## Imports

In [2]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib.mlab   as mlab

from xkte_nonadaptive import kernel_dr_two_sample_test_agnostic
from projected_adaptive_kte import projected_adaptive_kte_test
from sklearn.metrics import pairwise_distances

from scipy.spatial.distance import cdist
from scipy.special import expit
from scipy.stats import bernoulli
from numpy.polynomial.polynomial import polyval
from sklearn.linear_model import LogisticRegression

from scipy.stats import norm
import scipy.stats as stats
import statistics

from tqdm import tqdm

import seaborn as sns
import pandas as pd
import time
import os


In [3]:
import numpy as np

def treatment_effect_vector(ns, scenario, rng, beta_mix=2.0, beta_uniform=4.0):
    if scenario == 'I':
        return np.zeros(ns)
    if scenario == 'II':
        return np.full(ns, 2.0)
    if scenario == 'III':
        signs = rng.binomial(1, 0.5, ns) * 2 - 1
        return signs.astype(float) * beta_mix
    if scenario == 'IV':
        return rng.uniform(-beta_uniform, beta_uniform, ns)
    return np.zeros(ns)

h_sigmoidal = lambda x: np.log(np.abs(16 * x - 8) + 1) * np.sign(x - 0.5)

def generate_iid_data(
    ns, d, beta_vec, noise_var, scenario,
    rng=None, policy="logistic"
):
    rng = rng or np.random.RandomState(0)

    X = rng.randn(ns, d)
    base = h_sigmoidal(X @ beta_vec)
    delta = treatment_effect_vector(ns, scenario, rng)

    Y0 = base + noise_var * rng.randn(ns)
    Y1 = base + noise_var * rng.randn(ns) + delta

    # independent logging policy
    if policy == "logistic":
        logits = X @ beta_vec
        w = 1 / (1 + np.exp(-logits))       # p(a=1 | x)
    elif policy == "uniform":
        w = np.full(ns, 0.5)
    else:
        raise ValueError("policy must be 'logistic' or 'uniform'")

    A = (rng.rand(ns) < w).astype(int)
    Y = np.where(A == 1, Y1, Y0)

    # since i.i.d. we set folds arbitrarily:
    idx0 = np.arange(0, ns, 2)
    idx1 = np.arange(1, ns, 2)
    Pi_fold0_on_0 = np.tile(w[idx0], (len(idx0), 1))
    Pi_fold1_on_1 = np.tile(w[idx1], (len(idx1), 1))

    # full predictable propensities: same p(x) for all t (i.i.d.)
    P_all = np.tile(w, (ns, 1))

    return X, A, Y[:, None], w, Pi_fold0_on_0, Pi_fold1_on_1, idx0, idx1, P_all

## Main function, which runs a list of tests based on its arguments

In [4]:
import os, time
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import tqdm 
from tqdm import tqdm

import os, time
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

def run_tests_adaptive(
    b_list, method_list, ns_list,
    name_folder, num_experiments, iterations,
    seed=42,
    split="alternating"   # kept for filename suffix only
):
    noise_var = 0.1
    d = 5
    beta_vec = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
    os.makedirs(name_folder, exist_ok=True)

    rng = np.random.RandomState(seed)

    for b in b_list:
        print("b =", b)
        for method in method_list:
            for ns in tqdm(ns_list):
                p_values = np.zeros(num_experiments)
                values   = np.zeros(num_experiments)
                times    = np.zeros(num_experiments)

                for n in range(num_experiments):
                    policy = 'logistic'
                    experiment = True if (policy=='uniform') else False
                    X, T, Y, w, Pi_0_on_0, Pi_1_on_1, idx0, idx1, P_all = generate_iid_data(
                        ns=ns,
                        d=d,
                        beta_vec=beta_vec,
                        noise_var=noise_var,
                        scenario=b,
                        rng=rng,
                        policy=policy,   # or "uniform"
                    )

                    if Y.ndim == 1:
                        Y = Y[:, None]

                    YY0 = Y[T == 0]
                    YY1 = Y[T == 1]
                    sigma2 = np.median(
                        pairwise_distances(YY0, YY1, metric="euclidean")
                    )**2 / 4
                    if (not np.isfinite(sigma2)) or sigma2 <= 0:
                        sigma2 = float(np.var(Y)) + 1e-6

                    y = Y.reshape(-1)

                    t0 = time.time()

                    if method == "ADR-KTE":
                        out = projected_adaptive_kte_test(
                            Y=Y,
                            X=X,
                            A=T,
                            policy_matrix=P_all,
                            idx_pilot=idx0,
                            idx_infer=idx1,
                            outcome_gamma=1.0/sigma2,
                        )
                        value = out["statistic"]
                        p_value = out["p_value"]

                    elif method == "DR-xKTE":
                        value, p_value = kernel_dr_two_sample_test_agnostic(
                            Y, X, T, w,
                            kernel_function='rbf',
                            lam=5*1e-2,
                            gamma=1.0/sigma2,
                            verbose=False
                        )
                        from math import erf
                        p_value = 0.5 * (1.0 - erf(value / np.sqrt(2.0)))

                    elif method == "KTE":
                        stat, _, pval = kernel_two_sample_test_nonuniform(
                            YY0,
                            YY1,
                            T,
                            w.reshape(-1),
                            kernel_function="rbf",
                            iterations=iterations,
                            random_state=seed,
                            gamma=1.0 / sigma2,
                        )
                        value = stat
                        p_value = pval

                    
                    elif method == 'DR-CFME':
                        t0 = time.time()
                        stat, _, p_value = kernel_dr_nonuniform(Y, X, T, w, experiment, iterations=10, verbose=False, kernel_function='rbf', gamma=1.0/sigma2)
                        value=stat
                    
                    else:
                        raise ValueError("Method not recognized.")

                    times[n]    = time.time() - t0
                    p_values[n] = p_value
                    values[n]   = value

                df = pd.DataFrame(
                    {
                        "times": times,
                        "p_values": p_values,
                        "stat_values": values,
                    }
                )
                df.to_csv(
                    os.path.join(name_folder, f"ns{ns}b{b}{method}_{split}.csv"),
                    index=False,
                )


## Run functions for different settings

### Null hypothesis

In [5]:
num_experiments=200
iterations=100

ns_list = np.arange(100, 550, 50)
b_list = ['I']
method_list = ['ADR-KTE', 'DR-xKTE']
# method_list = ['DR-CFME']

experiment = 'iid_rebuttal'
name_folder = 'results/' + str(experiment) + '/'
run_tests_adaptive(b_list, method_list, ns_list, name_folder, num_experiments, iterations, split="alternating")

b = I


100%|██████████| 9/9 [14:19<00:00, 95.52s/it] 


## H1 scenarios II, III, IV

In [6]:
num_experiments=200
iterations=100

ns_list = np.arange(100, 550, 50)
b_list = ['II']

method_list = ['ADR-KTE', 'DR-xKTE']

experiment = 'iid_rebuttal'
name_folder = 'results/' +str(experiment) + '/'
run_tests_adaptive(b_list, method_list, ns_list, name_folder, num_experiments, iterations, split="alternating")

b = II


100%|██████████| 9/9 [00:12<00:00,  1.41s/it]


In [7]:
import os
import numpy as np
import pandas as pd

def aggregate_results_adaptive(
    name_folder="results/iid_rebuttal/",
    scenario_list=("I", "II"),
    ns_list_null=np.arange(100, 550, 50),   # from cell 7
    ns_list_alt=np.arange(100, 550, 50),    # from cell 12
    methods=("ADR-KTE", "DR-xKTE"),
    split_suffix="_alternating.csv",
    alpha=0.05,
):
    rows = []

    # Scenario I (null) uses ns_list_null; II–IV use ns_list_alt
    for scenario in scenario_list:
        if scenario == "I":
            ns_list = ns_list_null
        else:
            ns_list = ns_list_alt

        for method in methods:
            for ns in ns_list:
                fname = f"{name_folder}ns{ns}b{scenario}{method}{split_suffix}"
                if not os.path.exists(fname):
                    # skip missing files
                    continue

                df = pd.read_csv(fname)
                pvals = df["p_values"].values
                stat_vals = df["stat_values"].values
                times = df["times"].values

                rej = (pvals < alpha).mean()
                rows.append(
                    {
                        "scenario": scenario,
                        "ns": ns,
                        "method": method,
                        "rej_rate": rej,
                        "mean_stat": stat_vals.mean(),
                        "mean_time": times.mean(),
                        "n_experiments": len(pvals),
                    }
                )

    results_df = pd.DataFrame(rows)
    return results_df

results_table = aggregate_results_adaptive()
results_table

results_pivot = (
    results_table
    .pivot_table(
        index=["scenario", "ns"],
        columns="method",
        values="rej_rate",
    )
    .reset_index()
)

In [8]:
# --- Build the master results table ---
results_table = aggregate_results_adaptive()

# --- Produce one transposed table per scenario ---
scenario_tables = {}

for scenario in results_table["scenario"].unique():
    df_s = results_table[results_table["scenario"] == scenario]

    # Pivot: rows = methods, columns = ns
    table = (
        df_s.pivot_table(
            index="method",
            columns="ns",
            values="rej_rate",
        )
        .sort_index(axis=1)
        .sort_index(axis=0)
    )

    scenario_tables[scenario] = table

# Display all scenario-specific tables
for scenario, table in scenario_tables.items():
    print(f"\n===== Scenario {scenario} =====\n")
    display(table)



===== Scenario I =====



ns,100,150,200,250,300,350,400,450,500
method,,,,,,,,,
DR-xKTE,0.085,0.105,0.05,0.045,0.075,0.065,0.055,0.075,0.055
ADR-KTE,0.085,0.040,0.09,0.085,0.050,0.080,0.085,0.070,0.065



===== Scenario II =====



ns,100,150,200,250,300,350,400,450,500
method,,,,,,,,,
DR-xKTE,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
ADR-KTE,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
